Importing Basic Libraries

In [1]:
import pandas as pd
import numpy as np

Importing Regression Funcitons

In [2]:
from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

Importing Evaluator Functions

In [4]:
from sklearn.metrics import r2_score

from sklearn.metrics import mean_absolute_percentage_error
from sktime.performance_metrics.forecasting import median_absolute_percentage_error

Importing Classes and Functions from Alternate Files

In [5]:
from model_classes import W4_Regression, W5_Regs, W7_Boosting

# from baseline_model import W4_Regression
# from model_comparison import W5_Regs

# from advanced_models import W7_Boosting
# from advanced_models import main

In [6]:
# Preset file

filename = "CRMLS_0625-0626_enriched.csv"
end_mnth = 6

In [7]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [8]:
enr_df = load_df()

In [9]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols


cols = enr_df.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
log_cols = main_cols+new_cols


target = "ClosePrice"

In [10]:
class W8_Eval():

  def __init__(self,df):
    self.df = df


  def mape_eval(self, y_test, y_pred):
    mape = mean_absolute_percentage_error(y_test, y_pred)       # Computes the mean absolute percentage error of y_test and the predicted y
    return mape


  def mdape_eval(self, y_test, y_pred):
    mdape = median_absolute_percentage_error(y_test, y_pred)       # Computes the median absolute percentage error of y_test and the predicted y
    return mdape

In [11]:
base = W4_Regression(enr_df)
comp = W5_Regs(enr_df)
boost = W7_Boosting(enr_df)
Week8 = W8_Eval(enr_df)

train, test = base.test_train_split()


log_df = base.log_transform()

bs = W4_Regression(log_df)
cp = W5_Regs(log_df)
bst = W7_Boosting(log_df)
Wk8 = W8_Eval(log_df)

tr, te = bs.test_train_split()

# **Evaluation:  *MAPE, MdAPE***

 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

**Linear Regression**

In [12]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LRmape, LRmape = comp.shrt_main(base.LinReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmape, logLRmape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LRmdape, LRmdape = comp.shrt_main(base.LinReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmdape, logLRmdape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
['AvgAreaCost']:
 0.2703
Log Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgLivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ClosePrice/DaysOnMarket', 'FireplaceYN_False', 'FireplaceYN_True', 'BathroomsTotalInteger/BedroomsTotal', 'PoolPrivateYN_False', 'PoolPrivateYN_True', 'LivingArea/DaysOnMarket', 'PostalCode', 'DistrictID', 'LotSizeSquareFeet', 'NewConstructionYN_True', 'DaysOnMarket', 'YearBuilt', 'ViewYN_False', 'AvgAreaLot', 'ViewYN_True', 'NewConstructionYN_False', 'LotSizeSquareFeet/DaysOnMarket']:
 0.017


Median Absolute Percentage Error 
Non-Transform
['AvgAreaCost']:
 0.18
Log Transform
['AvgAreaCost', 'BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgLivingArea', 'FireplaceYN_False', 'FireplaceYN_True', 'BathroomsTotalInteger', 'BedroomsTotal', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'PoolPrivateYN_Fals

**Decision Tree Regressor**

In [12]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_DTRmape, DTRmape = comp.shrt_main(comp.TreeReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmape, logDTRmape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_DTRmdape, DTRmdape = comp.shrt_main(comp.TreeReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmdape, logDTRmdape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'LivingArea', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']:
 0.0017
Log Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'ClosePrice/DaysOnMarket', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'FireplaceYN_False']:
 0.0001


Median Absolute Percentage Error 
Non-Transform
['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'AvgAreaCost']:
 0.0
Log Transform
['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'AvgAreaCost', 'PostalCode', 'AvgAreaLot', 'AvgLivingArea', 'DistrictID', 'Close

**Random Forest Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_RFRmape, RFRmape = comp.shrt_main(comp.ForestReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmape, logRFRmape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_RFRmdape, RFRmdape = comp.shrt_main(comp.ForestReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmdape, logRFRmdape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform


**XGBoost**

In [18]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_XGBmape, XGBmape = comp.shrt_main(boost.XGB, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logXGBmape, logXGBmape = cp.shrt_main(bst.XGB, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_XGBmdape, XGBmdape = comp.shrt_main(boost.XGB, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logXGBmdape, logXGBmdape = cp.shrt_main(bst.XGB, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'PostalCode', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_True']:
 0.0113
Log Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'DistrictID', 'PostalCode', 'BathroomsTotalInteger/ClosePrice', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'YearBuilt', 'LotSizeSquareFeet/DaysOnMarket', 'PoolPrivateYN_True', 'DaysOnMarket']:
 0.0005


Medi

**Gradient Boosting Regressor**

In [19]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_GBRmape, GBRmape = comp.shrt_main(boost.GBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logGBRmape, logGBRmape = cp.shrt_main(bst.GBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_GBRdmape, GBRmdape = comp.shrt_main(boost.GBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)
c_logGBRdmape, logGBRmdape = cp.shrt_main(bst.GBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")

Mean Absolute Percentage Error
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'LivingArea', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'AvgAreaLot', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'FireplaceYN_False']:
 0.0436
Log Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LotSizeSquareFeet/DaysOnMarket']:
 0.0017


Median Absolute Percentage Error 
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'PostalCode', 'LivingArea', 'AvgLivingArea', 'BathroomsTot

**LightGBM**

In [21]:
# MAPE

In [ ]:
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LGBMmape, LGBMmape = comp.shrt_main(boost.L_GBM, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

In [ ]:
print("Log Transform")
c_logLGBMmape, logLGBMmape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

In [ ]:
# MdAPE

In [ ]:
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LGBMmdape, LGBMmdape = comp.shrt_main(boost.L_GBM, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

In [ ]:
print("Log Transform")
c_logLGBMmdape, logLGBMmdape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

***Mean Absolute Percentage Error***

*Non-Transform*

* ***0.0181***

*Log Transform*

* ***0.0007***

***Median Absolute Percentage Error***

*Non-Transform*

* ***0.01***

*Log Transform*

* ***0.0007***

**Histogram-based Gradient Boosting Regressor**

In [17]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_HGBRmape, HGBRmape = comp.shrt_main(boost.HGBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logHGBRmape, logHGBRmape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_HGBRmdape, HGBRmdape = comp.shrt_main(boost.HGBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logHGBRmdape, logHGBRmdape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

Mean Absolute Percentage Error
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False']:
 0.0173
Log Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet']:
 0.0008


Median Absolute Percentage Error 
Non-Transform
['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'PostalCode', 'DistrictID', 'BathroomsTotalInteger/ClosePrice', 'AvgLivingArea', 'AvgAreaLot', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'ClosePrice/DaysOnMarket', 'BedroomsTotal', 'LotS

***Summary***

 * Highest MAPE:  **Decision Tree**
   * Non-Transform: **.0017**
   * Log Transform: **.0001**

 * Highest MdAPE: **Decision Tree**
   * Non-Transform: **0.0**
   * Log Transform: **0.0**

# **Metrics DataFrame: *R2, MAPE, MdAPE***

 * R2 Score (R2)
 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

In [ ]:
modl = ["LinearRegression"]*4 + ["Decision Tree Regressor"]*4 + ["Random Forest Regressor"]*4 + [
    "XGBoost"]*4 + ["Gradient Boosting Regressor"]*4 + ["LightGBM"]*4 + [Histogram-based Gradient Boosting Regressor]*4

typ = ["MAPE", "MAPE", "MdAPE", "MdAPE"]*7

cols = [c_LRmape, c_logLRmape, c_LRmape, c_logLRmape] + [c_DTRmape, c_logDTRmape, c_DTRmdape, c_logDTRmdape] + [
    c_RFRmape, c_logRFRmape, c_RFRmdape, c_logRFRmdape] + [c_XGBmape, c_logXGBmape, c_XGBmdape, c_logXGBmdape] + [
        c_GBRmape, c_logGBRmape, c_GBRmdape, c_logGBRmdape] + [c_LGBMmape, c_logLGBMmape, c_LGBMmdape, c_logLGBMmdape] + [
        c_HGBRmape, c_logHGBRmape, c_HGBRmdape, c_logHGBRmdape]

scrs = [LRmape, logLRmape, LRmdape, logLRmdape] + [DTRmape, logDTRmape, DTRmdape, logDTRmdape] + [RFRmape, logRFRmape, RFRmdape, logRFRmdape] + [
    XGBmape, logXGBmape, XGBmdape, logXGBmdape] + [GBRmape, logGBRmape, GBRmdape, logGBRmdape] + [LGBMmape, logLGBMmape, LGBMmdape, logLGBMmdape] + [
        HGBRmape, logHGBRmape, HGBRmdape, logHGBRmdape]

In [ ]:
NA = [None]*28
mets = {"Model": modl, "ScoreType": typ, "ScoreValue": scrs, "max_depth": NA, "learning_rate": NA, "n_estimators": NA, "columns": cols}

met_df = pd.DataFrame(mets)

adv_results = main()
adv_results.insert()

met_df.concat(adv_results)

In [ ]:
# Preset File

metrics_df = "metrics_summary.csv"

In [ ]:
def save_csv(df, file=metrics_df):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
save_csv()